# Appendix

This notebook contains additional code used to generate some ancillary inputs for the demos in the other notebooks.

## Compute $\overline{\mathrm{SIF}_j}$ and $\overline{\Delta \mathrm{SIF}_j}$ for a Region of Interest

To derive the Z-scores and RCI for the SIF data used in the first notebook, we need to determine mean SIF over the entire time range of our source data for each j-th 8-day window. For completeness, we will download the entire GOSIF dataset from 2001-2020 and compute the mean value per grid cell over the region of interest from the notebook, saving the result in a CSV. The download step will take about 20 minutes.

**Note:** The primary complexity in the second cell is due to the fact that Mohammadi and Wang compute z-scores of SIF anomaly _per grid cell_ rather than spatially averaging the data over the region of interest first and then computing the mean and standard devation of the SIF increment. To reiterate what was mentioned in the first notebook, the z-score involves dividing by the standard deviation of SIF incremement which is not commutative:
$$
Z(j, y) = \frac{\Delta \mathrm{SIF}_{j, y} - \overline{\Delta \mathrm{SIF}_j}}{\sigma_{\Delta \mathrm{SIF}_j}}
$$

In other words, the value for $\sigma_{\Delta \mathrm{SIF}_j}$ when computed per grid cell and then spatially averaged will differ from the value that would be obtained by spatially averaging first and then computing $\sigma_{\Delta \mathrm{SIF}_j}$. It's not strictly necessary to do this for $\overline{\mathrm{SIF}_j}$, the value in the third band (we could spatially average the data now), but we'll leave it gridded for now since there's no harm in doing so.

In [ ]:
import os
from tqdm.notebook import tqdm
from download import download_unpack_gosif

start_year = 2001
end_year = 2020

dates: list[tuple[int, int]] = []
for year in range(start_year, end_year + 1):
    for doy in range(1, 365, 8):
        dates.append((year, doy))

output_dir = "data/gosif"
os.makedirs(output_dir, exist_ok=True)

gosif_geotiffs: list[str] = []
for date_tuple in tqdm(dates, desc="Downloading granules"):
    fname = download_unpack_gosif(
        date_tuple[0],
        day=date_tuple[1],
        output_dir=output_dir,
        verbose=False
    )
    if fname:
        gosif_geotiffs.append(fname)

In [ ]:
import contextlib
import os
import warnings
from glob import glob

import numpy as np
import rasterio
from rasterio.windows import from_bounds, transform as window_transform

input_dir = "data/gosif"
gosif_geotiffs = sorted(glob(f"{input_dir}/GOSIF_???????.tif"))
output_dir = "inputs/sif_increments"
os.makedirs(output_dir, exist_ok=True)

if "start_year" not in locals():
    start_year = 2001
if "end_year" not in locals():
    end_year = 2020

# Northern Great Plains region of interest
# 45.00°–50.00°N, 106.00°–111.00°W
west, south, east, north = -111.0, 45.0, -106.0, 50.0

# The threshold and scale factor parameters come from the documentation:
# https://data.globalecology.unh.edu/data/GOSIF_v2/Fair_Data_Use_Policy_and_Readme_GOSIF_v2.pdf
# 32767 = water bodies, 32766 = ice/snow
gosif_data_thresh = 32765
# This value tells our code the conversion between pixel values in the GeoTIFF images to units of W/m^2/sr/μm
gosif_scale_factor = 0.0001

N_WINDOWS = 46  # GOSIF 8-day windows per year: DOY 1, 9 ... 361
ddof = 1        # 1 = sample std, 0 = population

# Georeferencing of the ROI window (identical across all GOSIF files)
with rasterio.open(gosif_geotiffs[0]) as src:
    read_window = from_bounds(west, south, east, north, src.transform)
    read_window = read_window.round_offsets().round_lengths()
    win_transform = window_transform(read_window, src.transform)
    crs = src.crs
height = int(read_window.height)
width = int(read_window.width)


def read_roi(path: str) -> np.ndarray:
    """Read the ROI, mask non-data (water/ice/fill), scale to physical units."""
    with rasterio.open(path) as src:
        arr = src.read(1, window=read_window).astype("float64")
    arr[arr > gosif_data_thresh] = np.nan
    return arr * gosif_scale_factor

def parse_year_doy(path: str) -> tuple[int, int]:
    """GOSIF_YYYYDDD.tif -> (year, doy)."""
    name = os.path.splitext(os.path.basename(path))[0]
    token = name.split("_")[1]
    return int(token[:4]), int(token[4:7])

def window_index(doy: int) -> int:
    """Map a GOSIF day-of-year to a window index j"""
    return (doy - 1) // 8

def window_ordinal(year: int, j: int) -> int:
    """Global ordinal of an 8-day window. Consecutive windows differ by 1,
    including across year boundaries (year*46 + 45  ->  (year+1)*46 + 0)."""
    return year * N_WINDOWS + j

@contextlib.contextmanager
def _suppress_runtime_warnings():
    """nanmean/nanstd warn on all-NaN or <=1-sample pixels; those become NaN,
    which is the intended 'no climatology here' result."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        yield


# Step 1: accumulate increment grids per jth 8-day window
increments_by_j: dict[int, list[np.ndarray]] = {j: [] for j in range(N_WINDOWS)}
values_by_j: dict[int, list[np.ndarray]] = {j: [] for j in range(N_WINDOWS)}
prev_grid: np.ndarray | None = None
prev_ord: int | None = None

for path in gosif_geotiffs:
    year, doy = parse_year_doy(path)
    j = window_index(doy)
    ordi = window_ordinal(year, j)

    # read 'start_year - 1' as lead-in (to seed the j=0 increment) but
    # store nothing from it; ignore everything outside [start_year-1, end_year]
    if year < start_year - 1 or year > end_year:
        prev_grid, prev_ord = None, None
        continue

    grid = read_roi(path)
    if start_year <= year <= end_year:
        values_by_j[j].append(grid)

        # store only a genuinely consecutive increment assigned to the study period
        if prev_grid is not None and prev_ord == ordi - 1:
            increments_by_j[j].append(grid - prev_grid)

    prev_grid, prev_ord = grid, ordi

# Step 2: per-pixel mean/std across years, write one GeoTIFF per j
profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 3,
    "dtype": "float32",
    "crs": crs,
    "transform": win_transform,
    "nodata": np.nan,
    "compress": "deflate",
}

for j in range(N_WINDOWS):
    increment_list = increments_by_j[j]
    value_list = values_by_j[j]
    if not value_list or not increment_list:
        print(f"window j={j:02d} has no data, skipped")
        continue

    istack = np.stack(increment_list, axis=0)
    vstack = np.stack(value_list, axis=0)

    # per-pixel climatology; water/ice gaps ignored independently
    with np.errstate(invalid="ignore", divide="ignore"), _suppress_runtime_warnings():
        mean_incr = np.nanmean(istack, axis=0)
        std_incr = np.nanstd(istack, axis=0, ddof=ddof)
        mean_sif = np.nanmean(vstack, axis=0)

    doy_j = j * 8 + 1
    out_path = os.path.join(output_dir, f"GOSIF_dSIF_clim_{doy_j:03d}.tif")
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mean_incr.astype("float32"), 1)
        dst.write(std_incr.astype("float32"), 2)
        dst.write(mean_sif.astype("float32"), 3)
        dst.set_band_description(1, "mean dSIF")
        dst.set_band_description(2, "std dSIF")
        dst.set_band_description(3, "mean SIF")
        dst.update_tags(
            window_index=str(j),
            doy=str(doy_j),
            start_year=str(start_year),
            end_year=str(end_year),
            ddof=str(ddof),
            units="W/m^2/sr/um",
        )
    print(f"window j={j:02d} (DOY {doy_j:03d}): wrote {out_path}")